# Number Labyrinth — Full Board Generation

This notebook develops the complete board-generation algorithm:

1. **Path generation** — `generate_solution_path_dfs` from `generator_analysis.ipynb`.
2. **Path value assignment** — ascending sequence with random step in `[MIN_STEP, MAX_STEP]`.
3. **Non-path fill** — BFS wave-based flood-fill that propagates values outward from the path, using an A/B labelling scheme to control value direction.
4. **Display** — render completed boards with numbers and labels; highlight the solution path.

---

### Movement rule
A move from cell X to adjacent cell Y is allowed iff `value(X) < value(Y)`.  
A valid path from start `(0,0)` to goal `(cols-1, rows-1)` is a strictly-increasing walk.

### Fill algorithm overview
All path cells are initially labelled **A**.  
Each wave assigns labels to newly filled cells according to their numbered neighbours:

| # numbered neighbours | Condition | Value assigned | New label |
|---|---|---|---|
| 1 | neighbour is **A** or **B** | +step (prob P_HIGHER) / −step | A (if +) / B (if −) |
| 2 | both neighbours **A** | higher than both | B |
| 2 | at least one neighbour **B** | between the two (if room) | A |
| 2 | at least one neighbour **B**, no room between | smaller than both | B |
| 2 | at least one neighbour **B**, can't go smaller | larger than both | A |
| 3+ | any | smaller than all adjacent | B |

Step size sampled uniformly from `[MIN_STEP, MAX_STEP]` — the same range used for the path.

Uniqueness of the solution path is **not** guaranteed by this algorithm.

---
## 0 · Imports & global parameters

In [ ]:
import random
import math
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Tunable parameters ────────────────────────────────────────────────────────
MIN_STEP = 1    # minimum step size along path and for non-path value propagation
MAX_STEP = 20   # maximum step size
BASE_MIN = 1    # minimum starting value for the first path cell
BASE_MAX = 20   # maximum starting value for the first path cell
P_HIGHER = 0.5  # probability of going higher when first leaving the path (case 1)

# ── Level config ──────────────────────────────────────────────────────────────
LEVELS = {
    1: dict(cols=4, rows=5,  min_val=1, max_val=120),
    2: dict(cols=5, rows=5,  min_val=1, max_val=300),
    3: dict(cols=6, rows=6,  min_val=1, max_val=500),
    4: dict(cols=7, rows=7,  min_val=1, max_val=700),
    5: dict(cols=8, rows=8,  min_val=1, max_val=999),
}

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

---
## 1 · DFS solution-path generator

Copied verbatim from `generator_analysis.ipynb`.  
Returns one valid self-avoiding path from `(0,0)` to `(cols-1, rows-1)` satisfying the **neighbour-count rule** (intermediate cells have exactly 2 path-neighbours; endpoints have 1).

In [ ]:
def generate_solution_path_dfs(cols, rows, min_length=None, max_length=None, rng=None):
    """
    DFS single-path generator satisfying the neighbour-count rule.
    Returns a tuple of (col, row) pairs, or None if no path is found.
    """
    if rng is None:
        rng = random

    start = (0, 0)
    goal  = (cols - 1, rows - 1)

    def neighbors(c, r):
        return [(c+dc, r+dr) for dc, dr in DIRS
                if 0 <= c+dc < cols and 0 <= r+dr < rows]

    def manhattan(c, r):
        return abs(goal[0]-c) + abs(goal[1]-r)

    stack = [(start, frozenset([start]), (start,))]
    while stack:
        pos, visited, path = stack.pop()
        cands = list(neighbors(*pos))
        rng.shuffle(cands)
        for nc in cands:
            if nc in visited:
                continue
            if any(nb in visited and nb != pos for nb in neighbors(*nc)):
                continue
            new_len = len(path) + 1
            if max_length is not None and new_len + manhattan(*nc) > max_length:
                continue
            new_path    = path + (nc,)
            new_visited = visited | {nc}
            if nc == goal:
                if min_length is None or new_len >= min_length:
                    return new_path
            else:
                stack.append((nc, new_visited, new_path))
    return None


def get_path(cols, rows, rng=None):
    """Generate one path within the game length window [35%, 55%] of total cells."""
    T  = cols * rows
    lo = math.floor(0.35 * T)
    hi = math.floor(0.55 * T)
    return generate_solution_path_dfs(cols, rows, min_length=lo, max_length=hi, rng=rng)


# Smoke test
for lvl, cfg in LEVELS.items():
    p = get_path(cfg['cols'], cfg['rows'], rng=random.Random(lvl))
    print(f"Level {lvl}  ({cfg['cols']}×{cfg['rows']})  path len={len(p)}  ends at {p[-1]}")

---
## 2 · Assign values to path cells

$$
v_0 = \text{base},
\qquad
v_{i+1} = v_i + \delta_i,
\qquad
\delta_i \sim \mathrm{Uniform}(\texttt{MIN\_STEP},\, \texttt{MAX\_STEP})
$$

In [ ]:
def assign_path_values(path, min_step=None, max_step=None, base_val=None, rng=None):
    """
    Assign strictly increasing integer values to path cells.

    Returns
    -------
    dict  (col, row) -> int
    """
    if min_step is None: min_step = MIN_STEP
    if max_step is None: max_step = MAX_STEP
    if rng is None:      rng = random
    if base_val is None: base_val = rng.randint(BASE_MIN, BASE_MAX)

    values  = {}
    current = base_val
    for cell in path:
        values[cell] = current
        current += rng.randint(min_step, max_step)
    return values


# Demo
for lvl, cfg in LEVELS.items():
    rng  = random.Random(lvl * 100)
    path = get_path(cfg['cols'], cfg['rows'], rng=random.Random(lvl))
    pv   = assign_path_values(path, rng=rng)
    seq  = [pv[c] for c in path]
    steps = [seq[i+1]-seq[i] for i in range(len(seq)-1)]
    print(f"Level {lvl}  len={len(path):2d}  "
          f"values [{seq[0]}…{seq[-1]}]  "
          f"steps min={min(steps)} max={max(steps)} mean={sum(steps)/len(steps):.1f}")

---
## 3 · BFS wave fill — `fill_board`

### Full procedure

**Step 1.** Generate and number the solution path. Label every path cell **A**. All other cells are unnumbered.

**Step 2.** Let `numbered_fields` be the set of all cells that already have a value.  
Collect `neighbor_batch` = every unnumbered cell adjacent to `numbered_fields` (the current frontier).  
The membership of `numbered_fields` is **snapshotted** before processing the batch — cells assigned within the same wave do not affect each other.

**Step 3.** For each cell `f` in `neighbor_batch`:

- **If `f` is adjacent to exactly 1 cell** in `numbered_fields` (whether labelled A or B):  
  with prob `P_HIGHER` → assign `value + step`, label **A**;  
  otherwise → assign `value − step`, label **B**.

- **If `f` is adjacent to exactly 2 cells** in `numbered_fields`:  
  - **Case 1** — both are labelled **A**: assign a value higher than both, label **B**.  
  - **Case 2** — at least one is labelled **B**:  
    1. If there is an integer strictly between the two neighbours' values → pick one, label **A**.  
    2. Else if `global_min` allows going below both → assign smaller than both, label **B**.  
    3. Else → assign larger than both, label **A**.

- **If `f` is adjacent to 3 or more cells** in `numbered_fields`:  
  assign a value smaller than the minimum of all adjacent values, label **B**.

In all cases `step ~ Uniform(MIN_STEP, MAX_STEP)`. Values are clamped to `[global_min, global_max]`.

**Step 4.** Repeat Steps 2–3 until all cells are numbered.

In [ ]:
def fill_board(
    cols, rows, path, path_values,
    p_higher=None,
    min_step=None,
    max_step=None,
    global_min=1,
    global_max=999,
    rng=None,
):
    """
    Fill every non-path cell using the BFS wave labelling procedure.

    Labels
    ------
    'A'  — path cells (seeds) and non-path cells that went *higher* than their
            single numbered neighbour, or were placed between two neighbours
    'B'  — non-path cells that went *lower* than their single numbered neighbour,
            or higher than two A-neighbours, or smaller than 3+ neighbours

    Parameters
    ----------
    cols, rows   : board dimensions
    path         : sequence of (col, row) — solution path
    path_values  : dict (col,row)->int from assign_path_values()
    p_higher     : probability of +step when adjacent to a single numbered cell
    min_step     : minimum step size (default MIN_STEP)
    max_step     : maximum step size (default MAX_STEP)
    global_min   : floor for all assigned values
    global_max   : ceiling for all assigned values
    rng          : random.Random instance or None

    Returns
    -------
    board  : dict (col,row) -> int  — complete value assignment
    labels : dict (col,row) -> 'A' | 'B'
    """
    if p_higher is None: p_higher = P_HIGHER
    if min_step is None: min_step = MIN_STEP
    if max_step is None: max_step = MAX_STEP
    if rng      is None: rng      = random

    # Step 1: seed with path values; label every path cell A
    board  = dict(path_values)
    labels = {cell: 'A' for cell in path}

    def adj(c, r):
        return [(c+dc, r+dr) for dc, dr in DIRS
                if 0 <= c+dc < cols and 0 <= r+dr < rows]

    def sample_step():
        return rng.randint(min_step, max_step)

    def clamp(v):
        return max(global_min, min(global_max, v))

    # ── BFS wave loop ─────────────────────────────────────────────────────────
    while len(board) < cols * rows:

        # Step 2: snapshot + frontier
        snapshot = set(board.keys())
        neighbor_batch = {
            nc
            for cell in snapshot
            for nc in adj(*cell)
            if nc not in snapshot
        }
        if not neighbor_batch:
            break  # grid is fully covered

        # Step 3: assign each frontier cell
        for cell in neighbor_batch:
            numbered_nbrs = [n for n in adj(*cell) if n in snapshot]
            n_nbrs = len(numbered_nbrs)

            if n_nbrs == 1:
                # Single neighbour (A or B): probabilistic direction
                nbr_val = board[numbered_nbrs[0]]
                if rng.random() < p_higher:
                    new_val   = clamp(nbr_val + sample_step())
                    new_label = 'A'
                else:
                    new_val   = clamp(nbr_val - sample_step())
                    new_label = 'B'

            elif n_nbrs == 2:
                v0, v1 = board[numbered_nbrs[0]], board[numbered_nbrs[1]]
                l0, l1 = labels[numbered_nbrs[0]], labels[numbered_nbrs[1]]
                lo, hi = min(v0, v1), max(v0, v1)

                if l0 == 'A' and l1 == 'A':
                    # Case 1: both A → higher than both, label B
                    new_val   = clamp(hi + sample_step())
                    new_label = 'B'
                else:
                    # Case 2: at least one B → try between, else smaller, else larger
                    if hi - lo >= 2:
                        # room for an integer strictly between lo and hi
                        new_val   = rng.randint(lo + 1, hi - 1)
                        new_label = 'A'
                    elif clamp(lo - sample_step()) < lo:
                        # can go smaller than both
                        new_val   = clamp(lo - sample_step())
                        new_label = 'B'
                    else:
                        # fallback: go larger than both
                        new_val   = clamp(hi + sample_step())
                        new_label = 'A'

            else:
                # 3+ neighbours: value smaller than the minimum adjacent value, label B
                min_adj = min(board[n] for n in numbered_nbrs)
                new_val   = clamp(min_adj - sample_step())
                new_label = 'B'

            board[cell]  = new_val
            labels[cell] = new_label

    return board, labels


print("fill_board defined.")

### 3.1 · Sanity check — print one board with labels

In [ ]:
def print_board(cols, rows, board, path, labels=None):
    """Print the board as a text grid.  * marks path cells; A/B marks non-path labels."""
    path_set = set(path)
    print("    " + "".join(f"{c:6d}" for c in range(cols)))
    for r in range(rows):
        row_str = f"{r:2d}  "
        for c in range(cols):
            v = board[(c, r)]
            if (c, r) in path_set:
                marker = ' *'
            elif labels:
                lbl = labels.get((c, r), '?')
                marker = f' {lbl}' if lbl else '  '
            else:
                marker = '  '
            row_str += f"{marker}{v:4d}"
        print(row_str)
    print("  (* = path   A = higher / between   B = lower / higher-than-two-A)\n")


# ── Example: Level 1, seed 42 ─────────────────────────────────────────────────
rng  = random.Random(42)
cfg  = LEVELS[1]
path = get_path(cfg['cols'], cfg['rows'], rng=rng)
pv   = assign_path_values(path, rng=rng)
bd, lbl = fill_board(
    cfg['cols'], cfg['rows'], path, pv,
    global_min=cfg['min_val'], global_max=cfg['max_val'],
    rng=rng,
)

print(f"Level 1 (seed=42)  path={len(path)} cells  "
      f"v_start={pv[path[0]]}  v_goal={pv[path[-1]]}\n")
print_board(cfg['cols'], cfg['rows'], bd, path, lbl)

# Label distribution
from collections import Counter
non_path_labels = {k: v for k, v in lbl.items() if k not in set(path)}
cnt = Counter(non_path_labels.values())
print(f"Non-path label counts: {dict(cnt)}")
print(f"Value range on board:  [{min(bd.values())}–{max(bd.values())}]")

---
## 4 · Board display

Each cell shows its numeric value.  
Path cells (green) additionally show the step index (small, top) and value (bold, bottom).  
Non-path cells (beige) show the value at centre and a small label letter in the corner:
- **A** — went *higher* when first leaving the path (orange)
- **B** — went *lower* off path/A, or *higher* off another B (steel blue)

In [ ]:
LABEL_COLOR = {'A': '#c05000', 'B': '#1a4fa0'}

def display_board(cols, rows, board, path=None, labels=None, title=None, ax=None):
    """
    Render the full board with values and optional labels.

    Parameters
    ----------
    cols, rows : board dimensions
    board      : dict (col,row)->int
    path       : optional solution path list[(col,row)]
    labels     : optional dict (col,row)->None|'A'|'B' from fill_board_bfs
    title      : optional title string
    ax         : optional existing Axes
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(cols * 1.05, rows * 1.05))

    path_set   = set(path) if path else set()
    path_index = {cell: i for i, cell in enumerate(path)} if path else {}
    n_path     = len(path) if path else 0

    for r in range(rows):
        for c in range(cols):
            is_path = (c, r) in path_set
            checker = (c + r) % 2 == 0

            # ── Cell background ─────────────────────────────────────────────
            if is_path:
                face = '#b6e8b6' if checker else '#8fd98f'
            else:
                face = '#f5e6d0' if checker else '#e8d5b7'

            rect = mpatches.FancyBboxPatch(
                (c, rows - 1 - r), 1, 1,
                boxstyle='square,pad=0',
                linewidth=0.7, edgecolor='#aaa', facecolor=face,
            )
            ax.add_patch(rect)

            val = board.get((c, r), '')
            y0  = rows - 1 - r  # bottom-left y of cell in plot coords

            if is_path:
                # Step index — small, upper half
                idx = path_index[(c, r)]
                step_lbl = 'S' if idx == 0 else ('G' if idx == n_path - 1 else str(idx))
                ax.text(c + 0.5, y0 + 0.70, step_lbl,
                        ha='center', va='center', fontsize=6.5, color='#1a5c1a')
                # Value — bold, lower half
                ax.text(c + 0.5, y0 + 0.30, str(val),
                        ha='center', va='center', fontsize=9,
                        fontweight='bold', color='#0a3a0a')
            else:
                # Value — centre
                ax.text(c + 0.5, y0 + 0.50, str(val),
                        ha='center', va='center', fontsize=9, color='#5a3010')
                # Label — small, bottom-right corner
                if labels:
                    lbl = labels.get((c, r))
                    if lbl:
                        ax.text(c + 0.88, y0 + 0.12, lbl,
                                ha='center', va='center',
                                fontsize=6, fontweight='bold',
                                color=LABEL_COLOR.get(lbl, '#888'))

    # ── Path arrows ───────────────────────────────────────────────────────────
    if path:
        for i in range(len(path) - 1):
            c1, r1 = path[i];  c2, r2 = path[i + 1]
            ax.annotate(
                '', xy=(c2+0.5, rows-1-r2+0.5), xytext=(c1+0.5, rows-1-r1+0.5),
                arrowprops=dict(arrowstyle='->', color='#2a7a2a',
                                lw=1.3, shrinkA=14, shrinkB=14),
            )

    # ── Axis cosmetics ────────────────────────────────────────────────────────
    ax.set_xlim(0, cols);  ax.set_ylim(0, rows)
    ax.set_aspect('equal');  ax.axis('off')
    for c in range(cols):
        ax.text(c+0.5, rows+0.1, str(c), ha='center', va='bottom',
                fontsize=6.5, color='#999')
    for r in range(rows):
        ax.text(-0.1, rows-1-r+0.5, str(r), ha='right', va='center',
                fontsize=6.5, color='#999')
    if title:
        ax.set_title(title, fontsize=9, pad=4)
    if standalone:
        plt.tight_layout();  plt.show()

print("display_board defined.")

---
## 5 · Example boards — one per level (levels 1–5)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(26, 7), gridspec_kw={'wspace': 0.45})

for ax, (lvl, cfg) in zip(axes, LEVELS.items()):
    cols, rows = cfg['cols'], cfg['rows']
    rng  = random.Random(lvl * 7)
    path = get_path(cols, rows, rng=rng)
    pv   = assign_path_values(path, rng=rng)
    bd, lbl = fill_board(
        cols, rows, path, pv,
        global_min=cfg['min_val'], global_max=cfg['max_val'],
        rng=rng,
    )
    cnt = Counter(v for k, v in lbl.items() if k not in set(path))
    display_board(
        cols, rows, bd, path, lbl,
        title=(
            f"Level {lvl}  ({cols}×{rows})\n"
            f"path {len(path)} cells  [{pv[path[0]]}→{pv[path[-1]]}]\n"
            f"non-path: {cnt.get('A',0)}×A  {cnt.get('B',0)}×B"
        ),
        ax=ax,
    )

plt.suptitle(
    'Number Labyrinth — BFS fill  (orange A = higher/between, blue B = lower/over-two-A)',
    fontsize=12, y=1.02,
)
plt.show()

---
## 6 · Gallery — 6 random boards for a chosen grid size and P_HIGHER

Set `COLS`, `ROWS`, `P_HIGHER` and the value range at the top of the cell, then re-run.  
Each run draws 6 fresh random boards. Results are stored in the `gallery` dict (key = 0–5),  
each entry holding the seed used alongside the board data so any board can be reproduced.

In [ ]:
# ── Controls — edit these and re-run ─────────────────────────────────────────
COLS             = 5    # number of columns
ROWS             = 5    # number of rows
GALLERY_P_HIGHER = 0.5  # 0.0–1.0 — prob of going higher off a single numbered neighbour
GLOBAL_MIN       = 1    # floor for all cell values
GLOBAL_MAX       = 999  # ceiling for all cell values

# ── Generate 6 fresh random boards ───────────────────────────────────────────
gallery = {}   # key: 0–5 → dict with all specs + results
for i in range(6):
    seed = random.randint(0, 99_999)
    rng  = random.Random(seed)
    path = get_path(COLS, ROWS, rng=rng)
    pv   = assign_path_values(path, rng=rng)
    bd, lbl = fill_board(
        COLS, ROWS, path, pv,
        p_higher=GALLERY_P_HIGHER,
        global_min=GLOBAL_MIN, global_max=GLOBAL_MAX,
        rng=rng,
    )
    gallery[i] = dict(
        # specs
        cols     = COLS,
        rows     = ROWS,
        p_higher = GALLERY_P_HIGHER,
        global_min = GLOBAL_MIN,
        global_max = GLOBAL_MAX,
        seed     = seed,
        # results
        path         = path,
        path_values  = pv,
        board        = bd,
        labels       = lbl,
    )

# ── Display ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    2, 3,
    figsize=(COLS * 3.3, ROWS * 2.3),
    gridspec_kw={'hspace': 0.55, 'wspace': 0.38},
)

for i, ax in enumerate(axes.flat):
    g   = gallery[i]
    cnt = Counter(v for k, v in g['labels'].items() if k not in set(g['path']))
    display_board(
        g['cols'], g['rows'], g['board'], g['path'], g['labels'],
        title=(
            f"seed={g['seed']}\n"
            f"{len(g['path'])} path cells  "
            f"[{g['path_values'][g['path'][0]]}→{g['path_values'][g['path'][-1]]}]\n"
            f"{cnt.get('A', 0)}×A  {cnt.get('B', 0)}×B"
        ),
        ax=ax,
    )

fig.suptitle(
    f"{COLS}×{ROWS} grid  —  P_HIGHER={GALLERY_P_HIGHER}  —  6 random boards",
    fontsize=13, y=1.02,
)
plt.show()

# ── Print seed summary ────────────────────────────────────────────────────────
print(f"{'#':>2}  {'seed':>7}  {'path':>5}  {'A':>4}  {'B':>4}  value range")
for i, g in gallery.items():
    cnt   = Counter(v for k, v in g['labels'].items() if k not in set(g['path']))
    v_min = min(g['board'].values())
    v_max = max(g['board'].values())
    print(f"{i:>2}  {g['seed']:>7}  {len(g['path']):>5}  "
          f"{cnt.get('A',0):>4}  {cnt.get('B',0):>4}  [{v_min}–{v_max}]")

---
## 7 · Effect of P_HIGHER — label distribution vs. probability

`P_HIGHER` controls the probability of going *above* a path cell in the first wave.  
Here we sweep `p_higher` from 0.1 to 0.9 and measure the fraction of **A**-labelled non-path cells across 50 boards at Level 3.

In [ ]:
cfg  = LEVELS[3]
cols, rows = cfg['cols'], cfg['rows']
N_BOARDS   = 50
p_values   = [round(p, 1) for p in np.arange(0.1, 1.0, 0.1)]

results = {}  # p -> list of A-fractions
for p in p_values:
    fracs = []
    for seed in range(N_BOARDS):
        rng  = random.Random(seed)
        path = get_path(cols, rows, rng=rng)
        pv   = assign_path_values(path, rng=rng)
        bd, lbl = fill_board(
            cols, rows, path, pv,
            p_higher=p,
            global_min=cfg['min_val'], global_max=cfg['max_val'],
            rng=rng,
        )
        non_path = {k: v for k, v in lbl.items() if k not in set(path)}
        n_A = sum(1 for v in non_path.values() if v == 'A')
        fracs.append(n_A / len(non_path) if non_path else 0)
    results[p] = fracs

means  = [np.mean(results[p]) for p in p_values]
stds   = [np.std(results[p])  for p in p_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(p_values, means, yerr=stds, marker='o', capsize=4,
            color='#c05000', label='mean ± std')
ax.plot([0.1, 0.9], [0.1, 0.9], 'k--', lw=0.8, alpha=0.4, label='y = x (expected)')
ax.set_xlabel('P_HIGHER'); ax.set_ylabel('Fraction of A-labelled non-path cells')
ax.set_title(f'Level 3 ({cols}×{rows}) — A-label fraction vs P_HIGHER  (N={N_BOARDS} boards)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'P_HIGHER':>10}  {'mean A-frac':>12}  {'std':>8}")
for p, m, s in zip(p_values, means, stds):
    print(f"{p:10.1f}  {m:12.3f}  {s:8.3f}")

---
## 8 · Notes

### Label semantics recap
| Label | Value direction from neighbour | Next wave direction |
|-------|-------------------------------|---------------------|
| (none — path cell) | — seed | +step (prob P_HIGHER) / −step |
| **A** | +step above path neighbour | next gets −step → **B** |
| **B** | −step (from path/A) or +step (from B) | next gets +step → **B** |

The pattern: the first wave off the path can go either direction; once a cell is labelled A, its outward neighbours always dip back lower (label B); from B onward the wave rises again and stays B.  This creates a 'ridge-and-valley' texture around the path.

### Uniqueness
This fill does **not** guarantee a unique increasing path.  Where two A-labelled cells sit adjacent to consecutive path cells and their values fall in the right order, a shortcut can emerge.  Whether to fix this (e.g. by a post-process verification + re-draw of problematic cells) is left for a later step.

### Next steps
1. Measure how often alternative paths appear (BFS path-count on a sample of boards).
2. Optionally add a post-process pass to break any shortcuts found.
3. Tune `P_HIGHER`, `MIN_STEP`, `MAX_STEP` for desired difficulty per level.
4. Port the algorithm to `GameScene.js` once validated.